# Preprocess and upload dataset to HuggingFace (Phase 1)

In [ ]:
!pip install -U python-dotenv pandas numpy torch transformers accelerate bitsandbytes langchain-core huggingface_hub datasets pillow tqdm

**Phase 1.1:  Combine all cases into one .csv file**

In [1]:
import pandas as pd
import numpy as np
import json

from pathlib import Path, PurePosixPath

Locate custom dataset

In [ ]:
def find_dataset_root(dirname: str = "extracted_dataset_1", search_root: Path = None) -> Path:
    root = search_root or Path.cwd()

    direct = root / dirname
    if direct.is_dir():
        return direct

    matches = [p for p in root.rglob(dirname) if p.is_dir()]
    if matches:
        return matches[0]

    parent = root
    searched = [str(root)]
    for _ in range(4):
        parent = parent.parent
        searched.append(str(parent))
        matches = [p for p in parent.rglob(dirname) if p.is_dir()]
        if matches:
            return matches[0]

    raise FileNotFoundError(f"Could not find '{dirname}'. Searched under: {searched}")

DATASET_ROOT = find_dataset_root()
print("Using:", DATASET_ROOT)

Walk through each disease folder in **DATASET_ROOT**, and build one combined table with columns `case_id, case_text, disease`.

In [3]:
def build_combined_dataset(diseases: list[str], dataset_root: Path) -> pd.DataFrame:
    def process_image_metadata(image_metadata_path: Path):
        # image_metadata.json is JSON Lines (one dict per line), not a single JSON array
        case_id_to_images = {}
        with open(image_metadata_path, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                entry = json.loads(line)
                case_id = entry["case_id"]

                # raw file_path looks like "medical_datasets/<disease>/images/PMC5/PMC58/xxx.webp"
                # drop whatever the leading root segment is named, keep everything after it
                relative_parts = PurePosixPath(entry["file_path"]).parts[1:]
                resolved_path = str(dataset_root.joinpath(*relative_parts))

                # for every case_id, get a list of relevant images
                case_id_to_images.setdefault(case_id, []).append(resolved_path)

        return case_id_to_images

    all_rows = []

    for disease in diseases:
        disease_path = dataset_root / disease

        disease_df = pd.read_csv(disease_path / "cases_with_image.csv", dtype=str)
        image_paths = process_image_metadata(disease_path / "image_metadata.json")


        rows = pd.DataFrame({
            "case_id": disease_df["case_id"],
            "case_text": disease_df["case_text"],
            "disease": disease,
        })

        rows["images"] = rows["case_id"].map(lambda cid: image_paths.get(cid, []))

        n_missing = (rows["images"].str.len() == 0).sum()
        if n_missing:
            print(f"[warning] {disease}: {n_missing}/{len(rows)} cases had no matching image_metadata.json entry")

        all_rows.append(rows)

    all_cases_df = pd.concat(all_rows, ignore_index=True)

    return all_cases_df

In [ ]:
DISEASES = sorted([p.name for p in DATASET_ROOT.iterdir() if p.is_dir()])
all_cases_df = build_combined_dataset(DISEASES, DATASET_ROOT)
print(f"\nTotal combined: {len(all_cases_df)} rows across {all_cases_df['disease'].nunique()} diseases")

all_cases_df

**Phase 1.2:  Load AI model and prepare for preprocessing**

In [ ]:
import torch
import copy
import os

from langchain_core.prompts import PromptTemplate
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, DynamicCache

In [6]:
# Real variables: {case_text}, {disease}, {example_output_format}
INSTRUCTION_PROMPT = """
You are an expert clinician–educator.

You will receive a raw case report. Summarize the case into a JSON object for a high-quality vector database. Also make your own diagnosis from the text and justify it with reasoning and a direct quote from the case text. Follow the rules below strictly, without exceptions.

RULES (Read Carefully—No Exceptions)

1. Source Fidelity
- Use only facts from the case text.
- Do NOT invent, embellish, or “smooth out” missing data.
- Paraphrase prose into concise statements where helpful, but never add new facts.

2. Work Order
Work in this order: Metadata Skimming → Image Labeling → Case Presentation → Diagnostic Reasoning → Final Diagnosis.

3. JSON Keys
Use exactly these top-level keys: "patients", "case_presentation", "final_diagnosis", "diagnostic_reasoning".

4. "patients"
- "patients": one object per patient, each with "age" and "gender", as stated in the case text ("not stated" if missing). Use an empty list if no patient is mentioned.

5. "case_presentation"
- Read the case text again and summarize it, presenting only facts. Do not add outside knowledge, interpretation, or qualifiers (such as "remote" or "severe") that the case text does not use.
- The case text refers to images by filename (a name ending in ".webp"), usually wrapped in brackets or parentheses. Copy each filename and its surrounding brackets or parentheses exactly as written, character for character.
- Keep each image reference in the sentence of your summary that describes the finding it belongs to in the case text. Never group image references together and never move them to the end. Filenames that appear together in the case text (e.g., a pair of panels) stay together at that same position. Example: "CT showed a mass [<X_a.webp><X_b.webp>]. Later MRI showed edema [<Y_a.webp>]." stays exactly that way in the summary.
- Mention every image filename that appears in the case text. If you must cut words, cut other details before dropping a filename.
- References such as "Figure 2A-D" are not image filenames. Keep them as plain text if they matter, and never turn them into filenames.
- Copy numbers, units, and abbreviations exactly as written, even if they look like typos. Never convert or correct units.
- Attach a duration only to the symptom it was stated for. Keep the reference point of any timing (e.g., "postoperative day 5 from the first craniotomy").
- Prioritize the diagnostic workup (tests performed and their results, including negative and normal results) and the outcome and follow-up over procedural and operative detail. When you must cut words, cut procedural detail first. Never drop a negative or normal test result.
- Include the diagnosis as the case text states it.
- Keep the order of the case report (e.g., physical exam and labs before imaging).
- Stay as close as possible to the writing style of the case report.
- Aim for 250 to 300 words. Do not stop early while important workup findings or outcomes are still missing. Never exceed 300 words.

6. "final_diagnosis"
- A single disease/entity name (e.g., sarcoidosis).
- No adjectives, punctuation, or explanatory text.

7. "diagnostic_reasoning"
- "reasoning": one short sentence, at most 25 words, explaining how the case leads to final_diagnosis.
- "direct_quote": exactly one sentence, at most 30 words, copied exactly from the case text, that supports the reasoning. Do not combine sentences and do not paraphrase. Prefer a sentence that does not contain an image filename.

8. Output Format
- Output one valid JSON object and nothing else (no code fences, no extra text).
- Follow the template below exactly. Replace each UPPERCASE placeholder with real content, and repeat the patient object as many times as needed.
- Escape any double quotes inside strings.

OUTPUT_TEMPLATE_JSON (Copy exactly)

{
    "patients": [
          {"age": "PATIENT_AGE", "gender": "PATIENT_GENDER"}
    ]
    "case_presentation": "SUMMARY_OF_CASE_TEXT",
    "final_diagnosis": "DISEASE_NAME",
    "diagnostic_reasoning": {
        "reasoning": "SHORT_SENTENCE_FOR_REASONING",
        "direct_quote": "QUOTE_DIRECTLY_FROM_CASE_TEXT"
    }
}
"""

Load the model

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

try:
    from transformers import core_model_loading
    core_model_loading.GLOBAL_WORKERS = 1
except Exception:
    pass

MODEL_ID = "Qwen/Qwen3.5-4B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},   # keep everything on the GPU
)
model.eval()
print(f"Loaded {MODEL_ID} in 4-bit | GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB")

Set up KV cache for **SYS_PROMPT**

Relevant documentations:

Chat templates: https://huggingface.co/docs/transformers/main/en/chat_templating

KV Cache:   https://huggingface.co/docs/transformers/en/kv_cache

In [8]:
# Function to render query
def render(SYS_PROMPT, USER_PROMPT="No content", add_generation_prompt=True):

    MSGS = [{"role": "system", "content": SYS_PROMPT}, {"role": "user", "content": USER_PROMPT}]

    return tokenizer.apply_chat_template(
        MSGS,
        add_generation_prompt=add_generation_prompt,    # User tweaks behavior
        enable_thinking=False,                          # No <think></think> in response.
        tokenize=False,
        return_tensors="pt",
    )

In [9]:
def generate_cache(SYS_PROMPT):
    # For caching, disable generation
    END_OF_SYSTEM = "<|im_end|>\n"
    
    # Qwen requires a user turn, its content is irrelevant for caching
    DUMMY = render(SYS_PROMPT=SYS_PROMPT, add_generation_prompt=False)
    SYS_CUTOFF = DUMMY.index(END_OF_SYSTEM) + len(END_OF_SYSTEM)

    # System block ends right after <|im_end|>
    SYS_PROMPT_WITH_TOKENS = DUMMY[:SYS_CUTOFF]

    # Convert tokens
    SYS_TOKENS = tokenizer(SYS_PROMPT_WITH_TOKENS, return_tensors="pt", add_special_tokens=False).to("cuda")

    # Build KV cache from INSTRUCTION_TOKENS
    with torch.no_grad():
        SYS_CACHE = model(**SYS_TOKENS, past_key_values=DynamicCache(config=model.config), use_cache=True).past_key_values
        
    return SYS_CACHE

In [10]:
# Process multiple case prompts, extract image links and generate JSON for each of them
def generate_JSON_summary(SYS_PROMPT, USER_PROMPT):
    MAX_NEW_TOKENS = 1200

    QUERY = render(SYS_PROMPT=SYS_PROMPT, USER_PROMPT=USER_PROMPT, add_generation_prompt=True)
    QUERY_TOKENS = tokenizer(QUERY, return_tensors="pt", add_special_tokens=False).to("cuda")

    QUERY_OUTPUT = model.generate(
        **QUERY_TOKENS,
        past_key_values=copy.deepcopy(generate_cache(SYS_PROMPT)),
        max_new_tokens=MAX_NEW_TOKENS,                                           # Limit response to 800 tokens
        do_sample=False,
    )

    NEW_TOKENS = QUERY_OUTPUT[0, QUERY_TOKENS["input_ids"].shape[1]:]
    RESPONSE = tokenizer.decode(NEW_TOKENS, skip_special_tokens=True)

    # Convert response to real JSON
    try:
        RESULT, _ = json.JSONDecoder().raw_decode(RESPONSE[RESPONSE.find("{"):])
    except json.JSONDecodeError as e:
        TRUNCATED = NEW_TOKENS.shape[0] >= MAX_NEW_TOKENS
        raise ValueError(f"Model output is not valid JSON (hit token limit: {TRUNCATED}):\n{RESPONSE}") from e

    return RESULT

#TODO:  Run the **all_cases_df** above through the LLM, obtain a JSON summary for each case:

In [11]:
from functools import lru_cache
from tqdm.auto import tqdm

In [ ]:
USER_TEMPLATE = "Case Report: {case_report}"
LANGCHAIN_USER_TEMPLATE = PromptTemplate(input_variables=["case_report"], template=USER_TEMPLATE)

# generate_JSON_summary rebuilds the system-prompt KV cache on every call; build it once instead
generate_cache = lru_cache(maxsize=1)(generate_cache)

results = {}    # row index -> (json_text, error); lives in the kernel, so re-running the cell resumes

for idx, case_text in tqdm(all_cases_df["case_text"].items(), total=len(all_cases_df)):
    if idx in results and results[idx][0] is not None:
        continue                                              # already done
    user_prompt = LANGCHAIN_USER_TEMPLATE.format(case_report=case_text)
    try:
        result = generate_JSON_summary(SYS_PROMPT=INSTRUCTION_PROMPT, USER_PROMPT=user_prompt)
        results[idx] = (json.dumps(result, ensure_ascii=False), None)
    except ValueError as e:                                   # invalid or truncated JSON; message contains the raw output
        results[idx] = (None, str(e))
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        results[idx] = (None, "CUDA out of memory")

all_cases_df["json_text"] = pd.Series({i: r[0] for i, r in results.items()})
all_cases_df["error"] = pd.Series({i: r[1] for i, r in results.items()})
print(f"{all_cases_df['json_text'].notna().sum()}/{len(all_cases_df)} succeeded")

**Phase 1.3:  Split preprocessed data into train and test set**

In [13]:
def separate(df: pd.DataFrame, group_col: str = "disease", split_algorithm=None, seed: int = 42):
    if split_algorithm is None:
        split_algorithm = lambda test_proportion: 0.20        # default: always 20% test, regardless of group size

    rng = np.random.default_rng(seed)                         # default: rng seed 42
    train_parts, test_parts = [], []

    for group_val, group in df.groupby(group_col):
        n = len(group)
        test_proportion = split_algorithm(n)
        n_test = round(n * test_proportion)

        shuffled_idx = rng.permutation(group.index.to_numpy())
        test_idx = shuffled_idx[:n_test]
        train_idx = shuffled_idx[n_test:]

        test_parts.append(group.loc[test_idx])
        train_parts.append(group.loc[train_idx])

    train_df = pd.concat(train_parts, ignore_index=True)
    test_df = pd.concat(test_parts, ignore_index=True)

    return train_df, test_df

Separate dataset into train and test set, then upload both to HuggingFace repo

In [ ]:
def test_fraction_for(n: int) -> float:
    """Tiered test-set fraction based on how many cases a disease has."""
    if n >= 20:
        return 0.20
    elif n >= 5:  # 5 <= n < 20
        return 0.10
    else:  # n < 5
        return 0.0

train_df, test_df = separate(df=all_cases_df, split_algorithm=test_fraction_for, seed=69420)

print(f"\nTotal: train={len(train_df)} | test={len(test_df)}")
print(train_df)

**Phase 1.4:  Upload preprocessed dataset to HuggingFace**

In [ ]:
import os
from pathlib import Path
from huggingface_hub import login, whoami

In [ ]:
SECRET_NAME = "HF_TOKEN"        # key name in Colab Secrets and in the env file
ENV_FILE = "public.env"         # create a public.env with key "HF_TOKEN"

def token_from_colab(name: str) -> str:
    from google.colab import userdata          # raises ImportError outside Colab
    return userdata.get(name)                  # raises if secret is missing or access is denied


def token_from_envfile(name: str, path: str) -> str:
    from dotenv import dotenv_values           # pip install python-dotenv

    env_path = Path.cwd() / path
    if not env_path.is_file():
        raise FileNotFoundError(f"{env_path} does not exist. Did you forget to create the file?")

    value = dotenv_values(env_path).get(name)
    if not value or not value.strip():
        raise KeyError(f"'{name}' is missing or blank in {env_path}. Did you forget to add the variables?")
    return value.strip()


HF_TOKEN = None

# 1) Colab Secret
try:
    HF_TOKEN = token_from_colab(SECRET_NAME)
except Exception as e:
    print(f"Colab Secret not available ({type(e).__name__}), trying {ENV_FILE}...")

# 2) Fall back to the .env file (raises its own error if this fails too)
if not HF_TOKEN:
    HF_TOKEN = token_from_envfile(SECRET_NAME, ENV_FILE)

login(token=HF_TOKEN)
print(whoami())
os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
from datasets import Dataset, DatasetDict, Sequence, Image

In [16]:
def verify_image_paths(df: pd.DataFrame) -> None:
    missing = []
    for case_id, paths in zip(df["case_id"], df["images"]):
        for p in paths:
            if not Path(p).is_file():
                missing.append((case_id, p))
    if missing:
        print(f"[warning] {len(missing)} missing image files, e.g.:")
        for case_id, p in missing[:5]:
            print(f"   {case_id}: {p}")
    else:
        print("All image paths verified OK.")


def build_hf_dataset(df: pd.DataFrame) -> Dataset:
    ds = Dataset.from_pandas(df.reset_index(drop=True))
    ds = ds.cast_column("images", Sequence(Image()))  # path strings -> embedded images
    return ds

In [ ]:
verify_image_paths(train_df)
verify_image_paths(test_df)

REPO_ID = "<DATASET_LINK_GOES_HERE>"
dataset_dict = DatasetDict({
    "train": build_hf_dataset(train_df.reset_index(drop=True)),
    "test": build_hf_dataset(test_df.reset_index(drop=True)),
})

print(dataset_dict["train"].features["images"])

In [ ]:
dataset_dict.push_to_hub(REPO_ID, token=HF_TOKEN)
print(f"\nPushed to https://huggingface.co/datasets/{REPO_ID}")